Merge Data

In [6]:
import sys
print(sys.executable)


c:\Users\missm\OneDrive\Documents\Capstone_Project_Strickland\CP\Scripts\python.exe


In [ ]:
# Load Cleaned Datasets
import pandas as pd

kt1 = pd.read_parquet("KT1_cleaned_sample.parquet")
kt2 = pd.read_parquet("KT2_cleaned_sample.parquet")
kt3 = pd.read_parquet("KT3_cleaned_sample.parquet")
kt4 = pd.read_parquet("KT4_cleaned_sample.parquet")

print(kt1.shape, kt2.shape, kt3.shape, kt4.shape)


(1402362, 6) (6025501, 7) (9820184, 7) (15270789, 9)


In [ ]:
# Standardize each tag to avoid future confusion
kt1["dataset"] = "KT1"
kt2["dataset"] = "KT2"
kt3["dataset"] = "KT3"
kt4["dataset"] = "KT4"


In [3]:
# Ensure common columns across all datsets
    # Make sure each has 'timestamp', 'user_id'
for df in [kt2, kt3, kt4]:
    if "timestamp" not in df.columns:
        df["timestamp"] = pd.NaT
    else:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    if "user_id" not in df.columns:
        print("Missing user_id!")

# Add dummy columns where needed so schemas match (optional step for concat)
for col in ['question_id', 'elapsed_time', 'user_answer']:
    for df in [kt2, kt3, kt4]:
        if col not in df.columns:
            df[col] = pd.NA


In [4]:
# Merge datasets into unified log
full_log = pd.concat([kt1, kt2, kt3, kt4], ignore_index=True)
full_log.sort_values(by=["user_id", "timestamp"], inplace=True)
full_log.reset_index(drop=True, inplace=True)

full_log.to_parquet("EdNet_full_log.parquet")


In [9]:
#Merge with questions.csv to analyze correctness
questions = pd.read_csv(r"C:\Users\missm\OneDrive\Documents\Capstone_Project_Strickland\CapstoneProject_LearningPatterns\questions.csv")
full_log = full_log.merge(
    questions[["question_id", "correct_answer"]],
    on="question_id",
    how="left"
)
full_log["is_correct"] = full_log["user_answer"] == full_log["correct_answer"]



In [10]:
student_features = full_log.groupby("user_id").agg(
    total_questions=("question_id", "count"),
    avg_elapsed_time=("elapsed_time", "mean"),
    accuracy_rate=("is_correct", "mean"),
    num_responses=("user_answer", "count"),
    num_actions=("action_type", "count"),  # from KT2–4
    first_timestamp=("timestamp", "min"),
    last_timestamp=("timestamp", "max"),
).fillna(0)

# Compute engagement duration in days
student_features["active_days"] = (student_features["last_timestamp"] - student_features["first_timestamp"]).dt.days


C:\Users\missm\AppData\Local\Temp\ipykernel_54632\1328603863.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [12]:
student_features = pd.read_parquet("student_behavior_features.parquet")
